# PyTorch, task by task

Run the setup cell, then one cell per task with **Shift+Enter**. Lines marked `# ◀` are the ones to look at. Change a number and run a cell again to see what changes.

In [ ]:
# Setup: the names, and the helpers that draw the pictures. Run this cell first.
import os, urllib.request
if not os.path.exists("names.txt"):
    urllib.request.urlretrieve("https://hongik-aicourses.github.io/ml-demos/week09-mlp-lm/names.txt", "names.txt")

import math
import matplotlib.pyplot as plt
import torch
torch.set_num_threads(1)

def show_emb(E, mark=(), grad=None):
    """The table E: one point per character (vowels red, consonants blue, '.' green)."""
    E = E.detach(); mark = set(int(i) for i in mark)
    fig, ax = plt.subplots(figsize=(5, 4.2))
    for i, c in enumerate(chars):
        col = "green" if c == "." else ("crimson" if c in "aeiou" else "royalblue")
        ax.text(E[i, 0], E[i, 1], c, color=col, ha="center", va="center", fontsize=13,
                fontweight="bold" if i in mark else "normal",
                bbox=dict(boxstyle="circle", fc="#fff1c2", ec="orange") if i in mark else None)
    if grad is not None:   # each row moves against its gradient
        G = -grad.detach()
        ax.quiver(E[:, 0], E[:, 1], G[:, 0], G[:, 1], color="gray", angles="xy", width=0.004)
    pad = 0.3
    ax.set_xlim(E[:, 0].min() - pad, E[:, 0].max() + pad); ax.set_ylim(E[:, 1].min() - pad, E[:, 1].max() + pad)
    ax.axhline(0, color="#eee", zorder=0); ax.axvline(0, color="#eee", zorder=0)
    ax.set_title("the table E: one point per character", fontsize=11)
    plt.show()

def show_bars(p, ctx, y=None):
    """P(next character | context) for the 27 characters; the true next one in red."""
    p = p.detach()
    fig, ax = plt.subplots(figsize=(8, 2.6))
    cols = ["crimson" if y is not None and i == int(y) else "royalblue" for i in range(len(chars))]
    ax.bar(range(len(chars)), p, color=cols)
    ax.set_xticks(range(len(chars))); ax.set_xticklabels(chars)
    ax.set_title(f"P(next | '{ctx}')", fontsize=11)
    plt.show()

def show_curves(losses, val):
    """The minibatch loss (mean of 100 steps) and the validation loss, every 100 steps."""
    avg = [sum(losses[s:s + 100]) / len(losses[s:s + 100]) for s in range(0, len(losses), 100)]
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.plot([s + 50 for s in range(0, len(losses), 100)], avg, label="minibatch loss (mean of 100 steps)")
    ax.plot([100 * (i + 1) for i in range(len(val))], val, color="crimson", label="validation loss")
    ax.axhline(math.log(27), color="gray", ls="--", lw=1); ax.set_xlabel("step"); ax.legend()
    plt.show()

def show_lr(curves):
    """The loss on all training examples, every 50 steps, for each learning rate."""
    fig, ax = plt.subplots(figsize=(7, 3))
    for lr, c in curves.items():
        ax.plot([50 * (i + 1) for i in range(len(c))], c, label=f"lr = {lr}")
    ax.axhline(math.log(27), color="gray", ls="--", lw=1); ax.set_xlabel("step"); ax.legend()
    plt.show()

def show(label, *values, key=False):
    """Print 'label : value'; the line to look at is highlighted in yellow and marked with ◀."""
    text = " ".join(str(v) for v in values)
    first, *rest = text.split("\n")
    pad = " " * (len(label) + 3)
    line = f"{label} : {first}" + "".join("\n" + pad + r for r in rest)
    print(f"\033[1;30;43m{line}\033[0m  ◀" if key else line)


## 1. Let's turn the 240 names into training examples

- **What we need:** Numbers, not letters: an id for each character, and pairs of (three ids → the next id).
- **In PyTorch:** `torch.tensor`: an array with a shape and a dtype (`int64` for ids).

In [ ]:
# --- The names
import torch, math
import torch.nn as nn
import torch.nn.functional as F

with open("names.txt") as f:          # 240 first names, one per line  ◀
    names = sorted(set(f.read().split()))
show("first and last names", names[:8], "...", names[-3:])
g = torch.Generator().manual_seed(0)
perm = torch.randperm(len(names), generator=g).tolist()
n_tr = int(0.85 * len(names))          # 85% to train  ◀
train_words = [names[i] for i in perm[:n_tr]]
val_words = [names[i] for i in perm[n_tr:]]
show("names, training, validation", len(names), len(train_words), len(val_words), key=True)
show("first training names", train_words[:6])

# --- Characters and their ids
chars = ["."] + sorted(set("".join(names)))   # ◀
stoi = {c: i for i, c in enumerate(chars)}   # ◀
V = len(chars)
show("number of symbols V", V, key=True)
show("the symbols", chars)
show("ids of e, m, a", stoi["e"], stoi["m"], stoi["a"])

# --- The examples of one name
def make_examples(words, block=3):
    xs, ys = [], []
    for w in words:
        ctx = [0] * block                  # "..." at the start
        for ch in w + ".":
            xs.append(ctx); ys.append(stoi[ch])   # ◀
            ctx = ctx[1:] + [stoi[ch]]   # ◀
    return torch.tensor(xs), torch.tensor(ys)

X, Y = make_examples(["emma"])
show("X: contexts of emma", X, key=True)
show("Y: the next ids", Y, key=True)

# --- All the examples as tensors
Xtr, Ytr = make_examples(train_words)   # ◀
Xva, Yva = make_examples(val_words)
show("Xtr shape, dtype", Xtr.shape, Xtr.dtype, key=True)
show("Ytr shape, dtype", Ytr.shape, Ytr.dtype)
show("validation shapes", Xva.shape, Yva.shape)
show("first 4 contexts", Xtr[:4])

**What to look at**

- **Look at** `xs.append(ctx); ys.append(stoi[ch])` ◀: each example is (the last three ids, the next id); the context slides along.
  **Output:** emma gives 5 examples: X is 5 rows of 3 ids, Y the 5 next ids.
- **Look at** `Xtr, Ytr = make_examples(train_words)` ◀: the same for all 204 training names.
  **Output:** Xtr is (1378, 3) int64: 1,378 examples of 3 ids.

## 2. Let's give each character a vector

- **What we need:** A table with one learnable row per character: 27 rows of 2 numbers.
- **In PyTorch:** `nn.Embedding(27, 2)`: the table, and `emb(X)` looks up the rows of the ids in `X`.

In [ ]:
# --- The embedding table
torch.manual_seed(0)
emb = nn.Embedding(V, 2)               # 27 rows of 2 numbers  ◀
show("table shape", emb.weight.shape, key=True)
show("first 3 rows", emb.weight[:3].detach())
show_emb(emb.weight)

# --- Looking up rows
x = Xtr[:2]                            # two contexts
e = emb(x)                             # one row of the table per id  ◀
show("ids x", x)
show("shape of emb(x)", e.shape, key=True)
show("the rows", e.detach())
show_emb(emb.weight, mark=x.reshape(-1))

**What to look at**

- **Look at** `emb = nn.Embedding(V, 2)` ◀: one learnable row of 2 numbers per character.
  **Output:** shape (27, 2); random at the start, as the plot shows.
- **Look at** `e = emb(x)` ◀: calling the table on ids returns their rows.
  **Output:** ids of shape (2, 3) become vectors of shape (2, 3, 2).

## 3. Let's put the model in one object

- **What we need:** Something that holds all the parameters and turns a context into 27 scores.
- **In PyTorch:** `nn.Module`: the layers in `__init__`, the computation in `forward`; `softmax` turns scores into probabilities.

In [ ]:
# --- The model as a module
class MLPLM(nn.Module):
    def __init__(self, V=27, d=2, block=3, hidden=64):
        super().__init__()
        self.emb = nn.Embedding(V, d)   # ◀
        self.hidden = nn.Linear(block * d, hidden)   # ◀
        self.out = nn.Linear(hidden, V)   # ◀
    def forward(self, ctx):                         # ctx: (B, 3) ids  ◀
        e = self.emb(ctx).flatten(1)                # (B, 6)
        h = torch.tanh(self.hidden(e))              # (B, 64)
        return self.out(h)                          # (B, 27) scores

torch.manual_seed(0)
lm = MLPLM()
for name, p in lm.named_parameters():
    show(f"  {name}", tuple(p.shape))
show("parameters in all", sum(p.numel() for p in lm.parameters()), key=True)
show_emb(lm.emb.weight)

# --- Scores and probabilities
logits = lm(Xtr[:1])                   # the context "..."  ◀
p = F.softmax(logits, dim=1)   # ◀
show("scores shape", logits.shape)
show("first 5 scores", logits[0, :5].detach())
show("first 5 probabilities", p[0, :5].detach(), key=True)
show("sum of the 27", round(p.sum().item(), 4))
show_bars(p[0], '...', Ytr[0])

**What to look at**

- **Look at** `self.emb = nn.Embedding(V, d)` ◀: `__init__` makes the three layers; forward runs them in order.
  **Output:** 2257 numbers: all the parameters of the model.
- **Look at** `logits = lm(Xtr[:1])` ◀: the model gives 27 scores; softmax turns them into probabilities.
  **Output:** every probability is small, between 0.03 and 0.06, and they sum to 1.

## 4. Let's measure how wrong the model is

- **What we need:** $-\ln p$ of the character that really came next, averaged over a minibatch.
- **In PyTorch:** `F.cross_entropy(scores, targets)`: the softmax, $-\ln p$ and the average in one call.

In [ ]:
# --- The loss of one example
y = Ytr[:1]                            # the correct next character
show("next character, its p", chars[y.item()], round(p[0, y].item(), 4))
show("-ln p", round(-torch.log(p[0, y]).item(), 4))
show("F.cross_entropy", round(F.cross_entropy(logits, y).item(), 4), key=True)   # softmax, then -log  ◀
show_bars(F.softmax(logits, 1)[0], '...', Ytr[0])

# --- A minibatch
torch.manual_seed(1)
idx = torch.randint(0, len(Xtr), (64,))   # 64 random examples  ◀
show("first 10 example numbers", idx[:10])
show("minibatch shapes", Xtr[idx].shape, Ytr[idx].shape, key=True)

# --- The loss of the minibatch
loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])   # the mean over 64  ◀
show("loss of the minibatch", round(loss.item(), 4), key=True)
show("ln 27, a uniform guess", round(math.log(V), 4))

**What to look at**

- **Look at** `F.cross_entropy(logits, y)` ◀: the loss is −ln p of the correct next character.
  **Output:** the correct character c has p = 0.0302, so the loss is 3.4997.
- **Look at** `loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])` ◀: the same loss, averaged over the 64 examples.
  **Output:** 3.3321, close to ln 27 = 3.2958: the model still guesses.

## 5. Let's find how to change each parameter

- **What we need:** The gradient of the loss for every one of the 2,257 parameters.
- **In PyTorch:** `loss.backward()` fills `.grad` of every parameter. Gradients add up, so `zero_grad()` comes first.

In [ ]:
# --- Backpropagation
loss.backward()                        # fills .grad of every parameter  ◀
for name, p in lm.named_parameters():
    show(f"  {name}.grad", tuple(p.grad.shape), "size", round(p.grad.norm().item(), 4), key=name == "emb.weight")
used = lm.emb.weight.grad.abs().sum(1) > 0
show("rows with no gradient", [c for c, u in zip(chars, used) if not u])
show_emb(lm.emb.weight, grad=lm.emb.weight.grad)

# --- Gradients add up
g1 = lm.out.bias.grad.clone()
loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])
loss.backward()                        # a second time: added to .grad  ◀
show("after a second backward: twice the gradient?", torch.allclose(lm.out.bias.grad, 2 * g1), key=True)
opt = torch.optim.SGD(lm.parameters(), lr=0.3)
opt.zero_grad()                        # every .grad back to None  ◀
show(".grad after zero_grad()", lm.out.bias.grad)

**What to look at**

- **Look at** `loss.backward()` ◀: one call computes the gradient of every parameter.
  **Output:** each .grad has the shape of its parameter; the arrows show how E should move.
- **Look at** `opt.zero_grad()` ◀: a second backward() adds to .grad; zero_grad() clears it.
  **Output:** True: the gradient doubled; then None after zero_grad().

## 6. Let's take one step downhill

- **What we need:** $w \leftarrow w - \eta\,\nabla L$ for every parameter.
- **In PyTorch:** `torch.optim.SGD(lm.parameters(), lr=0.3)`; `opt.step()` does the update.

In [ ]:
# --- One step of SGD
loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])
opt.zero_grad()
loss.backward()
w = lm.emb.weight[stoi["e"]].detach().clone()
gw = lm.emb.weight.grad[stoi["e"]].clone()
opt.step()                             # w <- w - lr * grad  ◀
show("row of e, its gradient", w, gw)
show("w - 0.3 * grad", w - 0.3 * gw)
show("row of e after the step", lm.emb.weight[stoi["e"]].detach())
show("loss before, after", round(loss.item(), 3), round(F.cross_entropy(lm(Xtr[idx]), Ytr[idx]).item(), 3), key=True)
show_emb(lm.emb.weight, mark=[stoi['e']])

**What to look at**

- **Look at** `opt.step()` ◀: the update w ← w − 0.3 · grad, for every parameter.
  **Output:** the row of e moves a little, and the loss falls from 3.332 to 3.184.

---
# Training the model

## 7. Let's train the model

- **What we need:** The step, repeated 1,000 times, and the loss on names the model never trains on.
- **In PyTorch:** A Python loop; `torch.no_grad()` evaluates without recording a graph.

In [ ]:
# --- The training loop
losses, val = [], []
for step in range(1, 1001):
    idx = torch.randint(0, len(Xtr), (64,))
    loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])   # ◀
    opt.zero_grad()   # ◀
    loss.backward()   # ◀
    opt.step()   # ◀
    losses.append(loss.item())
    if step % 100 == 0:
        with torch.no_grad():          # no graph: we only evaluate
            val.append(F.cross_entropy(lm(Xva), Yva).item())
        show(f"step {step:4d}", f"minibatch loss {loss.item():.3f}   validation loss {val[-1]:.3f}", key=step == 1000)
show_curves(losses, val)

**What to look at**

- **Look at** `loss = F.cross_entropy(lm(Xtr[idx]), Ytr[idx])` ◀: the four lines of one step, repeated 1,000 times.
  **Output:** both losses fall; the validation loss is 2.674 at step 1,000.

## 8. Let's use the trained model

- **What we need:** The probabilities of the next letter, and new names drawn from them.
- **In PyTorch:** `F.softmax` for the probabilities; `torch.multinomial` draws one letter.

In [ ]:
# --- The next letter
ctx = torch.tensor([[stoi[c] for c in "emm"]])
with torch.no_grad():
    p = F.softmax(lm(ctx), dim=1)   # ◀
top = p[0].topk(5)
for q, i in zip(top.values, top.indices):
    show(f"  {chars[i]}", round(q.item(), 3), key=chars[i] == "a")
show_bars(p[0], 'emm')

# --- Sampling names
g = torch.Generator().manual_seed(1)
for _ in range(8):
    ctx, word = [0, 0, 0], ""
    while True:
        with torch.no_grad():
            p = F.softmax(lm(torch.tensor([ctx])), dim=1)
        i = torch.multinomial(p[0], 1, generator=g).item()   # ◀
        if i == 0 or len(word) > 15:
            break
        word += chars[i]; ctx = ctx[1:] + [i]
    show("  sampled", word, "(a training name)" if word in train_words else "")

**What to look at**

- **Look at** `p = F.softmax(lm(ctx), dim=1)` ◀: the probabilities of the letter after emm.
  **Output:** a is the most likely next letter, about 0.44.
- **Look at** `i = torch.multinomial(p[0], 1, generator=g).item()` ◀: draw a letter from p, append it, and repeat until '.'.
  **Output:** new names; a few are names from the training set.

## 9. Let's choose how long and how fast to train

- **What we need:** Compare runs: a longer one, and three learning rates.
- **In PyTorch:** The same loop, run again with other settings.

In [ ]:
# --- The learning rate
curves = {}
for lr in [0.03, 0.3, 3.0]:   # ◀
    torch.manual_seed(0)
    m = MLPLM()
    o = torch.optim.SGD(m.parameters(), lr=lr)
    curves[lr] = []
    for step in range(1, 1001):
        idx = torch.randint(0, len(Xtr), (64,))
        loss = F.cross_entropy(m(Xtr[idx]), Ytr[idx])
        o.zero_grad(); loss.backward(); o.step()
        if step % 50 == 0:
            with torch.no_grad():
                curves[lr].append(F.cross_entropy(m(Xtr), Ytr).item())
    show(f"  lr = {lr}", "loss after 1,000 steps", round(curves[lr][-1], 3), key=lr == 0.3)
show_lr(curves)

**What to look at**

- **Look at** `for lr in [0.03, 0.3, 3.0]:` ◀: the same training with three learning rates.
  **Output:** 0.3 ends lowest (2.139); 0.03 is slow (2.615); 3.0 jumps (2.374).